# Issues Faced and Resolutions

This notebook documents the issues encountered while integrating the Google ADK and Calendar APIs in the Calendar AI project, along with their solutions.

## 1. TypeError with `Runner.run()`

**Error Message:**
```python
TypeError: Runner.run() takes 1 positional argument but 2 were given
```

**Root Cause:**
This issue occurred when passing the prompt string as a positional argument to `runner.run()` (e.g., `await runner.run("prompt...")`). The `InMemoryRunner.run()` method does not accept the user input as a positional parameter (it expects keyword arguments for prompt/user input natively).

**Solution:**
Changed `runner.run()` to `runner.run_debug()`. `run_debug` natively accepts the prompt as the first positional argument, which perfectly matches our usage and provides helpful debug outputs as a bonus.

In [ ]:
# Code change in app.py
# Before:
# response = await runner.run("Can you list all the upcoming RCB matches...")

# After:
# response = await runner.run_debug("Can you list all the upcoming RCB matches...")

## 2. SSL/TLS Certificate `FileNotFoundError`

**Error Message:**
```python
ssl.create_default_context()
FileNotFoundError: [Errno 2] No such file or directory
```

**Root Cause:**
This is a common issue on Windows environments calling external HTTP APIs (like `google-genai` and `httpx`). Python's base `ssl` module tries to load trusted Certificate Authorities (CAs) to verify HTTPS connections but looks in standard OS locations that may not exist on a user's Windows machine.

**Solution:**
Explicitly set the `SSL_CERT_FILE` environment variable using Python's `certifi` package, which bundles reliable Mozilla root CA certs. Adding this configures the SSL client to resolve certificates properly.

In [ ]:
# Code change in agent.py
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

## 3. Agent Defensiveness and Search Tool Conflicts

**Error/Behavior:**
```text
search_agent > I am sorry, I cannot fulfill this request. I do not have access to your calendar...
```
When sending direct calendar commands (like "Delete the RCB match") into a `SequentialAgent` pipeline starting with a `search_agent`, the search agent panicked and replied defensively because it lacked calendar access. This defensive message cascaded down the pipeline, causing the parser and calendar agents to fail.

**Root Cause:**
1. **Sequential Blindness**: A rigid linear pipeline forces all commands through Search first, even if no search is required.
2. **Gemini Grounding Constraints**: The API restricts combining its native `google_search` grounding tool with local python custom tools (`delete_event_tool`) smoothly in a single unified agent context. We could not just merge them.

**Solution:**
Implemented a decoupled Director/Orchestrator pattern inside `app.py`. A lightweight LLM call dynamically evaluates the user's prompt and routes the traffic either to the `SequentialAgent` (for searching new events) or straight to the individual `calendar_agent` (for fast, direct modifications).

## 4. `gemini-2.5-flash-lite` Tool Execution Token Leak (`<ctrl42>` hallucination)

**Error/Behavior:**
```text
Part(text='<ctrl42>'),
Part(text='call\nprint(default_api.list_upcoming_events_tool())\n')
```
When running the agent to make a tool call, instead of emitting the officially structured `FunctionCall` protobuf, the model hallucinated raw pseudo-code filled with internal Gemini execution tokens (`<ctrl...>` strings) and dumped them into the terminal. This caused tool execution to hang or fail completely.

**Root Cause:**
The generic `gemini-2.5-flash-lite` model can sometimes "leak" internal code-execution/grounding structural tokens when trying to process complex agentic tools through the `google-adk` abstractions without perfect strict custom schema mapping.

**Solution:**
Globally upgraded all agent node definitions in `agent.py` and `app.py` directly to the `gemini-2.5-flash` (non-lite) model. The core `flash` model possesses much stronger native instruction-following and structured output alignment, effortlessly resolving the parsing and schema leakage issues natively.